In [ ]:
import argparse
import cv2
import math
import rasterio as rio
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader
from collections import OrderedDict

import sys 
sys.path.append('main-codes/Mamba-UNet-main/code')
from networks.vision_transformer import SwinUnet as ViT_seg
from networks.vision_mamba import MambaUnet as VIM_seg

sys.path.append("main-codes/modified_segmentation_models-main") 
import segmentation_models_pytorch as smp 

sys.path.append("main-codes/HybridModels")
from HybridSwinUNetC import HybridSwinUNetC
from HybridMambaUNetC import HybridMambaUNetC

from config import get_config

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
torch.cuda.empty_cache()

In [ ]:
# Argument Parser for Mamba-based models
mamba_parser = argparse.ArgumentParser()
mamba_parser.add_argument('--root_path', type=str, default='../data/ACDC', help='Name of Experiment')
mamba_parser.add_argument('--exp', type=str, default='ACDC/Fully_Supervised', help='experiment_name')
mamba_parser.add_argument('--model', type=str, default='mambaunet', help='model_name')
mamba_parser.add_argument('--num_classes', type=int, default=5, help='output channel of network')

mamba_parser.add_argument('--cfg', type=str, default="main-codes/Mamba-UNet-main/code/configs/vmamba_tiny.yaml", help='path to config file')
mamba_parser.add_argument('--opts', default=None, nargs='+', help="Modify config options by adding 'KEY VALUE' pairs.")

mamba_parser.add_argument('--zip', action='store_true', help='use zipped dataset instead of folder dataset')
mamba_parser.add_argument('--cache-mode', type=str, default='part', choices=['no', 'full', 'part'])
mamba_parser.add_argument('--resume', help='resume from checkpoint')
mamba_parser.add_argument('--accumulation-steps', type=int)
mamba_parser.add_argument('--use-checkpoint', action='store_true')
mamba_parser.add_argument('--amp-opt-level', type=str, default='O1', choices=['O0', 'O1', 'O2'])
mamba_parser.add_argument('--tag')
mamba_parser.add_argument('--eval', action='store_true')
mamba_parser.add_argument('--throughput', action='store_true')

mamba_parser.add_argument('--max_iterations', type=int, default=25)
mamba_parser.add_argument('--batch_size', type=int, default=8)
mamba_parser.add_argument('--deterministic', type=int, default=1)
mamba_parser.add_argument('--base_lr', type=float, default=0.0001)
mamba_parser.add_argument('--patch_size', type=int, nargs=2, default=[224, 224])
mamba_parser.add_argument('--seed', type=int, default=1337)
mamba_parser.add_argument('--labeled_num', type=int, default=140)

# Step 2: Simulated args for Jupyter/IPython
mamba_simulated_args = [
    '--root_path', '../data/ACDC',
    '--exp', 'ACDC/Fully_Supervised',
    '--model', 'mambaunet',
    '--num_classes', '5',
    '--cfg', 'main-codes/Mamba-UNet-main/code/configs/vmamba_tiny.yaml',
    '--patch_size', '224', '224',
    '--batch_size', '8',
    '--max_iterations', '25',
    '--base_lr', '0.0001',
    '--seed', '1337',
    '--labeled_num', '140'
]
mamba_args = mamba_parser.parse_args(args=mamba_simulated_args)

# Step 3: Load config
mamba_config = get_config(mamba_args)

In [ ]:
# Argument Parser for Swin-based models
swin_parser = argparse.ArgumentParser()
swin_parser.add_argument('--root_path', type=str, default='../data/ACDC', help='Name of Experiment')
swin_parser.add_argument('--exp', type=str, default='ACDC/Fully_Supervised_ViT', help='experiment_name')
swin_parser.add_argument('--model', type=str, default='unet', help='model_name')
swin_parser.add_argument('--max_iterations', type=int, default=25, help='maximum epoch number to train')
swin_parser.add_argument('--batch_size', type=int, default=8, help='batch_size per gpu')
swin_parser.add_argument('--deterministic', type=int, default=1, help='whether use deterministic training')
swin_parser.add_argument('--base_lr', type=float, default=0.0001, help='segmentation network learning rate')
swin_parser.add_argument('--patch_size', type=list, default=[224, 224], help='patch size of network input')
swin_parser.add_argument('--seed', type=int,  default=1337, help='random seed')
swin_parser.add_argument('--num_classes', type=int, default=5, help='output channel of network')
swin_parser.add_argument('--cfg', type=str, default="main-codes/Mamba-UNet-main/code/configs/swin_tiny_patch4_window7_224_lite.yaml", help='path to config file', )
swin_parser.add_argument('--opts', default=None, nargs='+', help="Modify config options by adding 'KEY VALUE' pairs. ")
swin_parser.add_argument('--zip', action='store_true',
                    help='use zipped dataset instead of folder dataset')
swin_parser.add_argument('--cache-mode', type=str, default='part', choices=['no', 'full', 'part'],
                    help='no: no cache, '
                         'full: cache all data, '
                         'part: sharding the dataset into nonoverlapping pieces and only cache one piece')
swin_parser.add_argument('--resume', help='resume from checkpoint')
swin_parser.add_argument('--accumulation-steps', type=int,
                    help="gradient accumulation steps")
swin_parser.add_argument('--use-checkpoint', action='store_true',
                    help="whether to use gradient checkpointing to save memory")
swin_parser.add_argument('--amp-opt-level', type=str, default='O1', choices=['O0', 'O1', 'O2'],
                    help='mixed precision opt level, if O0, no amp is used')
swin_parser.add_argument('--tag', help='tag of experiment')
swin_parser.add_argument('--eval', action='store_true',
                    help='Perform evaluation only')
swin_parser.add_argument('--throughput', action='store_true',
                    help='Test throughput only')

# label and unlabel
swin_parser.add_argument('--labeled_bs', type=int, default=12,
                    help='labeled_batch_size per gpu')
swin_parser.add_argument('--labeled_num', type=int, default=140,
                    help='labeled data')
# costs
swin_parser.add_argument('--ema_decay', type=float,  default=0.99, help='ema_decay')
swin_parser.add_argument('--consistency_type', type=str,
                    default="mse", help='consistency_type')
swin_parser.add_argument('--consistency', type=float,
                    default=0.1, help='consistency')
swin_parser.add_argument('--consistency_rampup', type=float,
                    default=200.0, help='consistency_rampup')

# Step 2: Simulated args for Jupyter/IPython
swin_simulated_args = [
    '--root_path', '../data/ACDC',
    '--exp', 'ACDC/Fully_Supervised_ViT',
    '--model', 'unet',
    '--num_classes', '5',
    '--cfg', 'main-codes/Mamba-UNet-main/code/configs/swin_tiny_patch4_window7_224_lite.yaml',
    '--patch_size', '[224, 224]',
    '--batch_size', '8',
    '--max_iterations', '25',
    '--base_lr', '0.0001',
    '--seed', '1337',
    '--labeled_num', '140'
]
swin_args = swin_parser.parse_args(args=swin_simulated_args)

# Step 3: Load config
swin_config = get_config(swin_args)

In [ ]:
testImg = ".../testImage.tif"   # Path to the test image

In [ ]:
def geoInfer(model_name,
             chckpnt,
             image_in, 
             pred_out, 
             chip_size, 
             stride_x, 
             stride_y, 
             crop, 
             n_channels):

    if model_name == "SwinUNet":
        model = ViT_seg(swin_config, 
                img_size=swin_args.patch_size,
                num_classes=swin_args.num_classes).to(device)
    elif model_name == "SwinUNetC":
        model = HybridSwinUNetC(swin_config, 
            img_size=swin_args.patch_size,
            num_classes=swin_args.num_classes).to(device)
    elif model_name == "MambaUNet":
        model = VIM_seg(mamba_config, 
                img_size=mamba_args.patch_size,
                num_classes=mamba_args.num_classes).to(device)
    elif model_name == "MambaUNetC":
        model = HybridMambaUNetC(mamba_config, 
                                    img_size=mamba_args.patch_size, 
                                    num_classes=mamba_args.num_classes).to(device)
    else:
        model = smp.Unet(
            encoder_name = "resnet34",
            encoder_weights = "imagenet",
            decoder_channels = (768, 768, 384, 192, 96),
            in_channels = 3,
            classes = 5
            ).to(device)

    print("The model is: ", model_name)

    # Load the state dict
    checkpoint = torch.load(chckpnt)

    # Remove `module.` prefix from keys
    new_state_dict = OrderedDict()
    for k, v in checkpoint.items():
        new_key = k.replace("module.", "")  # Remove "module." from the key
        new_state_dict[new_key] = v
    print(model.load_state_dict(new_state_dict))

    #Read in topo map and convert to tensor===========================
    image1 = cv2.imread(image_in)
    image1 = cv2.cvtColor(image1, cv2.COLOR_BGR2RGB)
    image1 = image1.astype('uint8')
    image1 = torch.from_numpy(image1)
    image1 = image1.permute(2, 0, 1)
    image1 = image1.float()/255
    t_arr = image1
    
    #Make blank grid to write predictions two with same height and width as topo===========================
    image2 = cv2.imread(image_in)
    image2 = cv2.cvtColor(image2, cv2.COLOR_BGR2RGB)
    image2 = image2.astype('uint8')
    image2 = torch.from_numpy(image2)
    image2 = image2.permute(2, 0, 1)
    image2 = image2.float()/255
    p_arr = image2[0, :, :]
    p_arr[:,:] = 0
    
    #Predict to entire topo using overlapping chips, merge back to original extent=============
    size = chip_size
    stride_x = stride_x
    stride_y = stride_y
    crop = crop
    n_channels = n_channels
    
    across_cnt = t_arr.shape[2]
    down_cnt = t_arr.shape[1]
    tile_size_across = size
    tile_size_down = size
    overlap_across = stride_x
    overlap_down = stride_y
    across = math.ceil(across_cnt/overlap_across)
    down = math.ceil(down_cnt/overlap_down)
    across_seq = list(range(0, across, 1))
    down_seq = list(range(0, down, 1))
    across_seq2 = [(x*overlap_across) for x in across_seq]
    down_seq2 = [(x*overlap_down) for x in down_seq]
    #Loop through row/column combinations to make predictions for entire image 
    for c in across_seq2:
        for r in down_seq2:
            c1 = c
            r1 = r
            c2 = c + size
            r2 = r + size
            #Default
            if c2 <= across_cnt and r2 <= down_cnt: 
                r1b = r1
                r2b = r2
                c1b = c1
                c2b = c2
            #Last column 
            elif c2 > across_cnt and r2 <= down_cnt: 
                r1b = r1
                r2b = r2
                c1b = across_cnt - size
                c2b = across_cnt + 1
            #Last row
            elif c2 <= across_cnt and r2 > down_cnt: 
                r1b = down_cnt - size
                r2b = down_cnt + 1
                c1b = c1
                c2b = c2
            #Last row, last column 
            else: 
                c1b = across_cnt - size
                c2b = across_cnt + 1
                r1b = down_cnt - size
                r2b = down_cnt + 1
            ten1 = t_arr[0:n_channels, r1b:r2b, c1b:c2b]
            ten1 = ten1.to(device).unsqueeze(0)
            model.eval()
            with torch.no_grad():
              ten2 = model(ten1)
            m = nn.Softmax(dim=1)
            pr_probs = m(ten2)              
            ten_p = torch.argmax(pr_probs, dim=1).squeeze(1)
            ten_p = ten_p.squeeze()
            #print("executed for " + str(r1) + ", " + str(c1))
            if(r1b == 0 and c1b == 0): #Write first row, first column
                p_arr[r1b:r2b-crop, c1b:c2b-crop] = ten_p[0:size-crop, 0:size-crop]
            elif(r1b == 0 and c2b == across_cnt+1): #Write first row, last column
                p_arr[r1b:r2b-crop, c1b+crop:c2b] = ten_p[0:size-crop, 0+crop:size]
            elif(r2b == down_cnt+1 and c1b == 0): #Write last row, first column
                p_arr[r1b+crop:r2b, c1b:c2b-crop] = ten_p[crop:size+1, 0:size-crop]
            elif(r2b == down_cnt+1 and c2b == across_cnt+1): #Write last row, last column
                p_arr[r1b+crop:r2b, c1b+crop:c2b] = ten_p[crop:size, 0+crop:size+1]
            elif((r1b == 0 and c1b != 0) or (r1b == 0 and c2b != across_cnt+1)): #Write first row
                p_arr[r1b:r2b-crop, c1b+crop:c2b-crop] = ten_p[0:size-crop, 0+crop:size-crop]
            elif((r2b == down_cnt+1 and c1b != 0) or (r2b == down_cnt+1 and c2b != across_cnt+1)): # Write last row
                p_arr[r1b+crop:r2b, c1b+crop:c2b-crop] = ten_p[crop:size, 0+crop:size-crop]
            elif((c1b == 0 and r1b !=0) or (c1b ==0 and r2b != down_cnt+1)): #Write first column
                p_arr[r1b+crop:r2b-crop, c1b:c2b-crop] = ten_p[crop:size-crop, 0:size-crop]
            elif (c2b == across_cnt+1 and r1b != 0) or (c2b == across_cnt+1 and r2b != down_cnt+1): # write last column
                p_arr[r1b+crop:r2b-crop, c1b+crop:c2b] = ten_p[crop:size-crop, 0+crop:size]
            else: #Write middle chips
                p_arr[r1b+crop:r2b-crop, c1b+crop:c2b-crop] = ten_p[crop:size-crop, crop:size-crop]
            
    #Read in a GeoTIFF to get CRS info=======================================
    image3 = rio.open(image_in)
    profile1 = image3.profile.copy()
    image3.close()
    profile1["driver"] = "GTiff"
    profile1["dtype"] = "uint8"
    profile1["count"] = 1
    profile1["PHOTOMETRIC"] = "MINISBLACK"
    profile1["COMPRESS"] = "NONE"
    
    pr_out = p_arr.cpu().numpy().round().astype('uint8')
    
    #Write out result========================================================
    with rio.open(pred_out, "w", **profile1) as f:
        f.write(pr_out,1)
    
    torch.cuda.empty_cache()

In [ ]:
MODELS_DIR = "" # Path to saved models
OUT_DIR = ""    # Path to save spatial predictions
arch = ["SwinUNet", "SwinUNetC", "MambaUNet", "MambaUNetC","UNet"]

for a in arch:
    geoInfer(model_name = a,
             chckpnt = MODELS_DIR + "Best_" + a + ".pt",
             image_in = testImg, 
             pred_out = OUT_DIR + a + "_pred.tif", 
             chip_size=224, 
             stride_x=112, 
             stride_y=112, 
             crop=16, 
             n_channels=3)